In [2]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import os
from pathlib import Path
import time as Time
import twstock
import yfinance
from datetime import datetime
from bs4 import BeautifulSoup
import random

In [ ]:
def finance_data(year,season,type="綜合損益表"):
    
    if season > 4 or season < 1:
        raise Exception("incorect season") 
    else:
        season = "0"+str(season)
    payload = {
        "encodeURIComponent": "1",
        "run": "",
        "step": "1",
        "TYPEK": "sii",
        "firstin": "true",
        "year": str(year),
        "season": season,
    }

    if type == "綜合損益表":
        url = "https://mops.twse.com.tw/mops/web/ajax_t51sb08"
    elif type == "資產負債表":
        url = "https://mops.twse.com.tw/mops/web/ajax_t51sb07"
        

    if year > 101 and type == "綜合損益表":
        url = "https://mops.twse.com.tw/mops/web/ajax_t163sb04"
    elif year > 101 and type == "資產負債表":
        url = "https://mops.twse.com.tw/mops/web/ajax_t163sb05"
        
    headers = {
        "Cookie":"jcsession=jHttpSession@9bf9964",
        "user-agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    }
    s = requests.Session()
    r = s.post(url=url, data=payload, headers=headers)
    r.encoding = 'utf8'

    dfs = pd.read_html(r.text,header=None)

    dfs = pd.concat(dfs[1:], axis=0, ignore_index=True)
    
    dfs = dfs.drop(columns=["合計：共 450 家"],errors="ignore")

    columns = dfs.columns 
    dfs = dfs.loc[:, ~dfs.columns.duplicated()] # TODO cant remove duplicate
    dfs.columns = columns
    dfs = dfs.rename(columns={'公司 代號':'公司代號'})
    if "公司代號" in dfs.columns:
        dfs = dfs.set_index("公司代號")
    else:
        raise KeyError("Column '公司代號' not found in the data.")
    return dfs   
# finance_data(89,1,type="資產負債表")

In [ ]:
def save_data(year:int, season:int, type:str):
    data = finance_data(year,season,type=type)
    curent_path = os.getcwd()
    base_path = Path(curent_path).resolve().parent
    file_name = f"{base_path}/{type}/{year+1911}Q{season}.csv"
    data.to_csv(file_name)
    print("save success")


# save_data(89,1,"綜合損益表")

In [ ]:
for year in range(108,114):
    for season in range(1,5):
        for table in ["綜合損益表","資產負債表"]:
            save_data(year,season,table)
            time.sleep(1)

股票代號

In [ ]:
set_twstock = set(stock for stock in twstock.twse)
set_twstock = list(filter(lambda x: (len(x)<5 and x[:2]!="00"), sorted(set_twstock)))
"1538" in set_twstock


In [ ]:
def save_ticker(ticker:str):
    try:
        now = datetime.now().strftime("%Y-%m-%d")
        start_date = "2000-1-1"
        data = yfinance.Ticker(ticker)
        df = data.history(start = start_date, end = now)
        if len(df.Close) == 0:
            return
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        file_name = f"{base_path}/上市股票歷史成交價/{ticker}.csv"
        df.to_csv(file_name)
        print(f"{ticker} save success")
    except Exception as e:
        raise e

In [ ]:
for stock in set_twstock:
    save_ticker(stock+".TW")

In [ ]:
def stock_interest(from_year:str,to_year:str,ticker:str):
    def type_check(year:str):
        if int(year) >1911:
            return str(int(year)-1911)
        return year
    from_year, to_year = type_check(from_year), type_check(to_year)
    payload = {
        "encodeURIComponent": "1",
        "step": "1",
        "firstin": "1",
        "off": "1",
        "keyword4": "",
        "code1": "",
        "TYPEK2": "",
        "checkbtn": "",
        "queryName": "co_id",
        "inpuType": "co_id",
        "TYPEK": "all",
        "isnew": "false",
        "co_id": ticker,
        "date1": from_year,
        "date2": to_year,
        "qryType": "1",
    }
    headers = {
        "Cookie":"jcsession=jHttpSession@67352714",
        "user-agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    }
    url = "https://mops.twse.com.tw/mops/web/ajax_t05st09_2"
    s = requests.Session()
    r = s.post(url=url, data=payload, headers=headers)
    r.encoding = 'utf8'

    dfs = pd.read_html(r.text,header=None)
    dfs = pd.concat(dfs[3:], axis=0, ignore_index=True)
    new_columns = dfs.columns.copy()
    new_columns = [col[0] for col in new_columns]
    dfs.columns = new_columns
    dfs = dfs.drop(columns=['摘錄公 司章程- 股利分 派部分', '備註'])
    return dfs
# stock_interest(2000,2024,"0050")

In [ ]:
def save_interest(ticker):
    
    try:
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        data = stock_interest(2000,2024,ticker)
        file_name = f"{base_path}/歷史股利發放政策/interest_{ticker}.TW.csv"
        data.to_csv(file_name)
        print("save success")
    except Exception as e:
        print(e)
        pass
# for ticker in set_twstock:
#     save_interest(ticker)
#     time.sleep(1)
# save_interest(2330)

In [ ]:
def modify_capital(year:int,season:int):
# /year = 2006
# season = 1
    try:
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        columns_keep = [ '公司名稱', '流動資產', '基金及投資', '固定資產', '無形資產', '其他資產', '流動負債', '長期負債', '其他負債',
            '股本', '資本公積', '保留盈餘', '其它項目', '資產總計', '負債總計', '股東權益', '每股淨值(元)',"預收股款（股東權益項下）之約當發行股數（單位：股）","預收股款（股東權益項下）之約當發行股數（單位：股）"]
        path = f"{base_path}/資產負債表/{year}Q{season}.csv"
        capital = pd.read_csv(path)
        if '每股淨值 (註一)'  in capital.columns:
            capital = capital.rename(columns={'每股淨值 (註一)':'每股淨值(元)'})
        

        # capital = capital[columns_keep]
        
        capital = capital.set_index(["公司代號"])
        if "公司 代號" in capital.index:
            capital.drop(index="公司 代號",inplace=True)
        capital.to_csv(path)
        print(f"{year}Q{season} capital modified")
    except Exception as e:
        print(f"{year}Q{season} {e}")
        pass
modify_capital(2006,1)

In [ ]:
for year in range(2000,2025):
    for season in range(1,5):
        modify_capital(year,season)

In [ ]:
def modify_revenue(year:int,season:int):
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/綜合損益表/{year}Q{season}.csv"
    revenue = pd.read_csv(path)
    revenue = revenue.set_index(["公司代號"])
    revenue.drop(index="公司 代號",inplace=True)
    del_columns = revenue.columns[-1] if "合計" in revenue.columns[-1] else []
    revenue.drop(columns=del_columns,inplace=True)
    revenue.to_csv(path)
    print(f"{year}Q{season} revenue modified")

In [ ]:
columns = set()
for year in range(2000,2025):
    for season in range(1,5):
        try:
            curent_path = os.getcwd()     
            base_path = Path(curent_path).resolve().parent
            path = f"{base_path}/綜合損益表/{year}Q{season}.csv" 
            df = pd.read_csv(path)
            for col in df.columns:
                if col.isdigit() or "Unnamed" in col:
                    break
                columns.add(col)
        except:
            pass

columns.add("年度")
columns.add("季度")
columns


In [85]:

total_df = pd.DataFrame(columns=list(columns))

In [ ]:
data = {}
for year in range(2000,2025):
    for season in range(1,5):
        try:
            curent_path = os.getcwd()     
            base_path = Path(curent_path).resolve().parent
            path = f"{base_path}/綜合損益表/{year}Q{season}.csv" 
            df = pd.read_csv(path)
            
            for row in range(len(df.get("公司代號"))):
                for col in total_df.columns:
                    row_data = df.loc[row]
                    value = row_data.get(col,np.nan)
                    if value == col or value == "公司 代號":
                        break
                    if data.get(col) is None:
                        data[col] = []
                    if col == "年度"  :
                        data[col].append(year)
                        continue
                    elif col == "季度":
                        data[col].append(f"Q{season}")
                        continue
                    data[col].append(value)
        
        except Exception as e: 
            print(e)
            
data


In [ ]:
total_df = pd.DataFrame(data)
total_df[(total_df["年度"]==2006) & (total_df["公司代號"]==1101)]["營業收入"]

In [136]:

total_df.to_csv(f"{base_path}/綜合損益表/合併綜合損益表.csv",index=False)


In [ ]:
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
path = f"{base_path}/上市股票歷史成交價/0050.TW.csv"
price = pd.read_csv(path)
price_columns = ["Ticker"]+list(price.columns)
price_columns


In [ ]:
price_data = {}
for ticker in set_twstock:
    try:
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        path = f"{base_path}/上市股票歷史成交價/{ticker}.TW.csv" 
        df = pd.read_csv(path)
        length = len(df["Close"])

        price_data["Ticker"] = price_data.get("Ticker",[]) + [ticker for _ in range(length)]
        nan_list = [np.nan for _ in range(length)]
        for col in price_columns[1:]:
            get_list = price_data.get(col,[])
            price_data[col] = get_list + list(df.get(col,nan_list))
    except Exception as e:
        print(f"{ticker} {e}")

In [ ]:
for key in price_data.keys():
    print(key, len(price_data[key]))

In [ ]:
price_df = pd.DataFrame(price_data)

In [ ]:
price_df.to_csv(f"{base_path}/上市股票歷史成交價/HistoryPriceFrom2000.csv")

In [ ]:
interest_col = set()
lost_ticker = []
for ticker in set_twstock:
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/歷史股利發放政策/interest_{ticker}.TW.csv" 
    if not os.path.exists(path):
        lost_ticker.append(ticker)
        continue
    try:
        df = pd.read_csv(path)
        for col in df.columns:
            if "Unnamed" in col:
                continue
            interest_col.add(col)
    except Exception as e:
        print(e)
        pass




In [ ]:
for ticker in lost_ticker:
    save_interest(ticker)
    time.sleep(5)

In [ ]:
interest_df = {}
for ticker in set_twstock:
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/歷史股利發放政策/interest_{ticker}.TW.csv" 
    if not os.path.exists(path):
        # lost_ticker.append(ticker)
        continue
    try:
        df = pd.read_csv(path)
        length = len(df.get("股利所屬 年(季)度"))
        interest_df["Tickers"] = interest_df.get("Tickers",[]) + [ticker for _ in range(length)]
        for col in interest_col:
            nan_list = [np.nan for _ in range(length)]
            df_col = list(df.get(col,nan_list) )
            if col in ['決議（擬議） 進度','期 別','股利所屬 期間','普通股 每股面額']:
                continue
            if col == '股利所屬 年(季)度':                
                year , season= [],[]
                for cont in df_col:
                    cont = cont.split(" ")
                    the_year = int(cont[0].replace("年",""))+1911
                    year.append(the_year)
                    if "季" in cont[1]:
                        the_season = int(cont[1][1])
                        season.append(the_season)
                    else:
                        season.append(np.nan)
                interest_df["Year"] = interest_df.get("Year",[]) + year
                interest_df["Season"] = interest_df.get("Season",[]) + season
            elif col == '董事會決議 (擬議)股 利分派日':
                distrib_date = []
                for cont in df_col: 
                    if isinstance(cont,float):
                        distrib_date.append(np.nan)
                        continue               
                    d_year, month, day= map(int,cont.split("/"))
                    d_year += 1911
                    date = datetime(d_year,month,day)
                    date = datetime.strftime(date,"%Y/%m/%d")
                    distrib_date.append(date)
                interest_df["股利分派日"] = interest_df.get("股利分派日",[]) + distrib_date
            elif col == '股東會 日期':
                meeting_date = []
                for cont in df_col:
                    if isinstance(cont,float) or (isinstance(cont,str) and not cont[0].isdigit()):
                        meeting_date.append(np.nan)
                        continue
                    d_year, month, day= map(int,cont.split("/"))
                    d_year += 1911
                    date = datetime(d_year,month,day)
                    date = datetime.strftime(date,"%Y/%m/%d")
                    meeting_date.append(date)
                interest_df["股東會日期"] = interest_df.get("股東會日期",[]) + meeting_date

            else:                        
                interest_df[col] = interest_df.get(col,[]) + df_col
        
    except Exception as e:
        print(ticker,e)
        pass

interest_df

In [ ]:
interest_dataframe = pd.DataFrame(interest_df)
interest_dataframe

In [ ]:
interest_dataframe.to_csv(f"{base_path}/歷史股利發放政策/historyinterestfrom2000.csv")

In [ ]:
# 爬取財報發布日期
def get_publish_date(ticker:str, year:int):
    if year > 1000:
        year -= 1911
    sess = requests.Session()
    url = "https://mops.twse.com.tw/mops/web/ajax_t57sb01_q1"
    payload = {
        "encodeURIComponent": "1",
        "step": "1",
        "firstin": "1",
        "off": "1",
        "TYPEK": "all",
        "keyword4":"" ,
        "code1": "",
        "TYPEK2": "",
        "checkbtn": "",
        "queryName": "co_id",
        "inpuType": "co_id",
        "co_id": ticker,
        "year": str(year),
    }
    headers = {
        "Cookie":"_cfuvid=ZVva1TY.q5e.xzvyrDwauRLUHSGjSQOKc1Pxy4w5wD0-1735181288434-0.0.1.1-604800000; _ga_J2HVMN6FVP=GS1.1.1735265061.4.0.1735265061.60.0.0; _gid=GA1.3.846327121.1736136400; _ga_5XRGGGWBYX=GS1.3.1736136400.1.0.1736136400.0.0.0; _ga=GA1.1.1747665681.1733886612; _ga_LTMT28749H=GS1.1.1736219719.17.1.1736219721.0.0.0",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        "content-type":"application/x-www-form-urlencoded"
    }
    try:
        # 進入網頁後爬取第二網址在進入
        response = sess.post(url=url, data=payload, headers=headers)
        html_parser = BeautifulSoup(response.text, "html.parser")
        second_url = html_parser.find("input")["value"].split("'")[1].strip("&")
        response = sess.get(url=second_url, headers=headers)
        html_parser = BeautifulSoup(response.text, "html.parser")
        df_list = pd.read_html(response.text)
        # 回傳有效表格
        for i in range(len(df_list)):
            if df_list[i].get("資料年度") is not None:
                return df_list[i]
        # 金融股財報會有第三層網頁
        third_url = "https://doc.twse.com.tw" +html_parser.find("form")["action"]  
        # 蒐集隱藏的input payload   
        payload_list = html_parser.find_all("input")
        payload_list = list(filter(lambda x: x["type"]=="hidden", payload_list))
        third_payload = { pl["name"] : pl["value"] for pl in payload_list }
        # 提交圖片的滑鼠點擊位置
        third_payload["x"] = str(random.randint(0,92))
        third_payload["y"] = str(random.randint(0,32))
        
        response = sess.post(url=third_url, data=third_payload, headers=headers)       
        df_list = pd.read_html(response.text)
        # 等待網頁回應
        Time.sleep(0.5)
        for i in range(len(df_list)):
            if df_list[i].get("資料年度") is not None:
                return df_list[i]        

    except Exception as e:
        print(ticker,year,e)



In [ ]:
# 上市股票財報發布日期爬蟲開始
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
frame = {
    "Ticker":[],
    "Year":[],
    "Season":[],
    "Publish_date":[],
}
iO_counter = 0
sleep_time = 5

for tik, ticker in enumerate(set_twstock):
    for year in range(2000,2025):
        df = get_publish_date(ticker,year)
        iO_counter += 1
        print("IO count:",iO_counter)        
        for idx, season in enumerate(["第一季","第二季","第三季","第四季"]):
            try:
                # 找到每季的資料上傳日期，將日期轉換格式 datetime64
                upload = list(df[df["資料年度"].str.contains(season, na=False)].head(1)["上傳日期"])[0]
                date, time = upload.split(" ")
                d_year,month,day = map(int,date.split("/"))
                # 民國年轉西元年
                d_year += 1911
                hour, min, sec = map(int,time.split(":"))
                new_date = datetime(d_year,month,day,hour,min,sec)
                # 暫存於frame
                frame["Ticker"].append(ticker)
                frame["Year"].append(year)
                frame["Season"].append(idx+1)
                frame["Publish_date"].append(new_date)
            except Exception as e:
                print(ticker,year,season,e)
                continue
        sleep_time = 5 - ((tik+1) % 3)
        print("Sleep time:", sleep_time)
        Time.sleep(sleep_time)
    # 存成csv
    new_df = pd.DataFrame(frame)    
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    new_df.to_csv(path)
    frame = {
        "Ticker":[],
        "Year":[],
        "Season":[],
        "Publish_date":[],
    }
        

In [ ]:
lost_ticker = {}

In [ ]:
def is_ascending(series:list, diff:int):
    if series == [] or series is None:
        return None
    series = sorted(series)
    for i in range(1,len(series)):
        if (series[i] - series[i-1]) != diff:
            return False
        return True
    
for ticker in set_twstock:
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    df = pd.read_csv(path)
    col_year = sorted(list(set(df.get("Year"))))
    diff = 1
    if not is_ascending(col_year, diff):
        lost_ticker[ticker] = lost_ticker.get(ticker,set())
        for i in range(1,len(col_year)):
            gap = int(col_year[i] - col_year[i-1])
            if gap != diff:
                # print("gap:",gap, "type:", type(gap))
                for j in range(1,gap):
                    lost_ticker[ticker].add(col_year[i-1] + j)

for ticker in set_twstock:
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    df = pd.read_csv(path)
    col_year = list(df.get("Year"))
    set_cy = set(col_year)
    for y in set_cy:
        if col_year.count(y) != 4:
            lost_ticker[ticker] = lost_ticker.get(ticker,set())
            lost_ticker[ticker].add(y)
lost_ticker
# empety = set()
# for key, value in lost_ticker.items():
#     if len(value) == 0:
#         empety.add(key)
# empety

In [ ]:
for tik, ticker in enumerate(empety):

    for year in range(2000,2025):
        df = get_publish_date(ticker,year)
        iO_counter += 1
        print("IO count:",iO_counter)        
        for idx, season in enumerate(["第一季","第二季","第三季","第四季"]):
            # pd.Series type transform
            try:
                upload = list(df[df["資料年度"].str.contains(season, na=False)].head(1)["上傳日期"])[0]
                date, time = upload.split(" ")
                d_year,month,day = map(int,date.split("/"))
                d_year += 1911
                hour, min, sec = map(int,time.split(":"))
                new_date = datetime(d_year,month,day,hour,min,sec)
                frame["Ticker"].append(ticker)
                frame["Year"].append(year)
                frame["Season"].append(idx+1)
                frame["Publish_date"].append(new_date)
            except Exception as e:
                print(ticker,year,season,e)
                continue
        sleep_time = 5 - ((tik+1) % 3)
        print("Sleep time:", sleep_time)
        Time.sleep(sleep_time)
    new_df = pd.DataFrame(frame)
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    new_df.to_csv(path)
    frame = {
        "Ticker":[],
        "Year":[],
        "Season":[],
        "Publish_date":[],
    }

In [ ]:
patch_frame = {
    "Ticker":[],
    "Year":[],
    "Season":[],
    "Publish_date":[],
}
iO_counter = 0
sleep_time = 2

for tik, (ticker, years) in enumerate(lost_ticker.items()):
    if len(years) == 0:
        years = [y for y in range(2000,2024)]
    for t_year in years:
        df = get_publish_date(ticker,t_year)
        counter = 1
        while (df is None or df.get("資料年度") is None) and counter <= 3:
            print(ticker,t_year, f"is None, connect {counter} times")
            Time.sleep(0.5)
            df = get_publish_date(ticker,t_year)
            counter += 1
            iO_counter += 1

        iO_counter += 1
        print("IO count:",iO_counter)        
        for idx, season in enumerate(["第一季","第二季","第三季","第四季"]):
            # pd.Series type transform
            try:
                upload = list(df[df["資料年度"].str.contains(season, na=False)].head(1)["上傳日期"])[0]
                # print(upload)
                date, time = upload.split(" ")
                d_year,month,day = map(int,date.split("/"))
                d_year += 1911
                hour, min, sec = map(int,time.split(":"))
                new_date = datetime(d_year,month,day,hour,min,sec)
                patch_frame["Ticker"].append(ticker)
                patch_frame["Year"].append(t_year)
                patch_frame["Season"].append(idx+1)
                patch_frame["Publish_date"].append(new_date)
            except Exception as e:
                print(ticker,t_year,season,e)
                continue
        sleep_time = 4 - ((tik+1) % 3)
        print("Sleep time:", sleep_time)
        Time.sleep(sleep_time)
    patch_df = pd.DataFrame(patch_frame).reset_index(drop=True)
    curent_path = os.getcwd()     
    base_path = Path(curent_path).resolve().parent
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    df = pd.read_csv(path)
    df = pd.concat([df, patch_df], ignore_index=True)
    df = df.sort_values(by=["Year", "Season"], ascending=[True, True])
    del_col = [col for col in df.columns if "Unnamed" in col]
    df = df.drop(columns=del_col).drop_duplicates()
    df.to_csv(path,index=False)
    patch_frame = {
        "Ticker":[],
        "Year":[],
        "Season":[],
        "Publish_date":[],
    }
    

合併財報數據以及歷史股價
1. 先將財報公布時間與損益表的所需數據整合成一個DF
2. merge_asof 兩份報告

In [ ]:
# 開啟合併損益表，過濾數據
modified_dict = {
    "date":[],
    "company_id":[],
    "revenue":[],
    "net_profit":[],
    "EPS":[],
    "Shareholders_equity":[],
}
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
revenue_path = f"{base_path}/綜合損益表/合併綜合損益表.csv"
revenue_df = pd.read_csv(revenue_path)
capital_path = f"{base_path}/資產負債表/合併資產負債表.csv"
capital_df = pd.read_csv(capital_path)
# loop 開啟每份日期，將需要的數據合併成一份表
def get_publish_date(ticker:str, book:dict, revenue_df:pd.DataFrame, capital_df:pd.DataFrame) -> dict:
    def chose_valid_col(candidates:list, series:dict) -> str:
        for candi in candidates:
            if not pd.isna(series[candi]):
                return candi
        return ""
    try:
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
        df = pd.read_csv(path)
        df = df.drop_duplicates()
        years = set(df["Year"])

        for y in years:
            siery = df[df["Year"]==y]
            
            for i in range(len(siery)):
                slc_siery = dict(siery.iloc[i])    
                               
                revenue_siery = revenue_df[(revenue_df["年度"] == int(y)) & (revenue_df["季度"] == f'Q{int(slc_siery.get("Season"))}')]
                capital_siery = capital_df[(capital_df["年度"] == int(y)) & (capital_df["季度"] == f'Q{int(slc_siery.get("Season"))}')]
                if not revenue_siery.empty and not capital_siery.empty:
                    revenue_siery = dict(revenue_siery.iloc[0])
                    capital_siery = dict(capital_siery.iloc[0])
                else:
                    continue
                # 損益表欄目名稱一共換過3次，需要找出該年的各項數據寫在那些欄目中
                revenue = revenue_siery.get(chose_valid_col(["營業收入 淨額", "營業收入"], revenue_siery))
                net_profit = revenue_siery.get(chose_valid_col(["稅後純益", "本期淨利（淨損）","本期淨利(淨損)"], revenue_siery))                             
                EPs = revenue_siery.get(chose_valid_col(["每股稅後盈餘(元)", "基本每股盈餘","基本每股盈餘（元）","每股盈餘"], revenue_siery))
                equity = capital_siery.get(chose_valid_col(["股東權益","股東權益總計","權益總額","權益總計"], capital_siery))
                # 資料暫存在dict
                book["date"].append(slc_siery.get("Publish_date"))
                book["company_id"].append(ticker)
                book["revenue"].append(revenue), book["net_profit"].append(net_profit), book["EPS"].append(EPs)
                book["Shareholders_equity"].append(equity)
                     
        return book
    except Exception as e:
        print(ticker,e)
# loop開始過濾company_id 傳入function，將所有數值儲存在modified_dict
for ticker in set_twstock:
    modified_dict = get_publish_date(ticker, modified_dict, revenue_df[revenue_df["公司代號"]==int(ticker)], capital_df[capital_df["公司代號"]==int(ticker)])

#儲存資料 
modified_DataFrame = pd.DataFrame(modified_dict)
modified_DataFrame.to_csv(f"{base_path}/modified_revenue.csv",index=False)

# get_publish_date("1538", modified_dict, revenue_df[revenue_df["公司代號"]==int(1538)], capital_df[capital_df["公司代號"]==int(1538)])

In [ ]:
a = revenue_df[revenue_df["公司代號"]==int(1538)]
a[(a["年度"]==2000) & (a["季度"]=="Q1")]

In [51]:
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
financials_path =  f"{base_path}/modified_revenue.csv"
stocks_path = f"{base_path}/上市股票歷史成交價/HistoryPriceFrom2000.csv"
# 讀取財報數據
financials = pd.read_csv(financials_path)
financials['date'] = pd.to_datetime(financials['date'])

# 讀取股票數據
stocks = pd.read_csv(stocks_path)
stocks['Date'] = pd.to_datetime(stocks['Date']).dt.tz_localize(None)

# 過濾無效欄位
del_unname = [col for col in stocks.columns if "Unnamed" in col][0]
stocks = stocks.drop(columns = [del_unname,"Dividends","Stock Splits","Capital Gains"])

# 確保日期格式統一
stocks = stocks.rename(columns={'Date': 'date', "Ticker":"company_id"})

# 將成交量為零的交易日剔除並排序
stocks = stocks[stocks["Volume"] != 0]
financials = financials.sort_values(by=['date'])
stocks = stocks.sort_values(by=['date'])


# 使用 merge_asof 合併，方向為 'forward'，確保財報數據合併到下一個交易日
merged = pd.merge_asof(
    stocks,
    financials,
    by='company_id',  # 基於公司 ID 合併
    on='date',        # 基於日期合併
    direction='forward'  # 向前合併到下一個交易日
)

# 將結果儲存
merged.to_csv('merged_financial_stock_data.csv', index=False)



In [ ]:
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
modi = pd.read_csv(f"{base_path}/modified_revenue.csv")
modi["revenue"] = pd.to_numeric(modi["revenue"], errors="coerce").fillna(0)
modi["growth_rate"] = round((
    modi.groupby("company_id")["revenue"].pct_change()
).fillna(0),4)
modi["Shareholders_equity"].fillna(0,inplace=True)
modi["net_profit"].fillna(0,inplace=True)
modi["ROE"] = round(modi["net_profit"]/modi["Shareholders_equity"]*100,4)
modi.to_csv(f"{base_path}/modified_revenue.csv",index=False)

需要再取得ROE的值
**ROE=淨利潤/股東權益**
從資產負債表取得**股東權益**


In [ ]:
revenue_df[revenue_df["公司代號"]==1538 ].to_csv("1538look.csv")